# 00 - Dataset audit, leakage check and split definitions

**Run this first.** Every later notebook loads the split definitions this one writes; nothing else
works until it has run once.

**What this notebook does**
1. Indexes every image and counts classes (and compares the total against the paper's stated 900).
2. Hashes every file (MD5 over raw bytes) and quantifies duplicates and cross-split leakage.
3. Writes two split definitions: `faithful` (the dataset's own folders, duplicates left in) and
   `clean` (deduplicated, leakage-free, group-aware stratified re-split).
4. Creates the three results CSVs with **headers only** - no numbers are ever hand-written.

**Lessons applied here**: 6 (leakage), 7 (class imbalance on `clean`), 8 (the 900-vs-1000 image
discrepancy).

**Cost**: no training. Hashing ~1000 files takes well under a minute on CPU.

**What "looks right"**: ~1000 images, four classes, ~153 duplicate files heavily concentrated in
`normal`, and a `clean` split that passes the no-leakage assertion.

In [ ]:
import sys, os
sys.path.append(os.path.abspath(".."))

from src.config import *
from src.data_utils import resolve_data_root

print(ensure_dirs())
print()
for k, v in describe_environment().items():
    print(f'{k}: {v}')
print('\ndata root:', resolve_data_root())

## 1. Index every image

**Looks right**: ~1000 rows, four classes, and `orig_split` covering train/val/test. The `folder`
column shows the dataset's inconsistent naming (long staging names in train/valid, short ones in
test) - `normalize_class_name()` has already mapped both onto the canonical class list.

In [ ]:
import numpy as np
import pandas as pd

from src.data_utils import index_dataset, class_counts

df = index_dataset()
print('images indexed:', len(df))
print()
print(df[['filename', 'class', 'label', 'orig_split', 'folder']].head(3).to_string(index=False))
print()
print('distinct source folders per class:')
print(df.groupby('class')['folder'].nunique().reindex(CLASS_NAMES).to_string())

In [ ]:
counts = class_counts(df)
print(counts.to_string())
print()
print('total images:', int(counts['total'].sum()))

## 2. The 900-vs-1000 discrepancy (LESSON 8)

The paper reports **900** images; this dataset now has ~**1000**. The cell below shows where the
difference sits. The expected finding, already established in the earlier attempts: the three tumour
classes match the paper exactly and the entire surplus is in `normal` - which is also where almost
all of the duplicates are. Most likely the dataset was expanded with additional (largely duplicate)
normal-class scans after publication.

**This needs no further investigation** - just state it in the README and move on.

In [ ]:
PAPER_CLASS_TOTALS = {   # as reported in the paper
    'adenocarcinoma': 338,
    'large.cell.carcinoma': 187,
    'squamous.cell.carcinoma': 260,
    'normal': 115,
}

cmp = pd.DataFrame({
    'ours': counts['total'],
    'paper': pd.Series(PAPER_CLASS_TOTALS).reindex(CLASS_NAMES),
})
cmp['difference'] = cmp['ours'] - cmp['paper']
print(cmp.to_string())
print()
print(f"total ours={int(cmp['ours'].sum())} paper={int(cmp['paper'].sum())} "
      f"difference={int(cmp['difference'].sum())}")
print()
surplus_in_normal = int(cmp.loc['normal', 'difference'])
total_surplus = int(cmp['difference'].sum())
if total_surplus and surplus_in_normal == total_surplus:
    print('CONFIRMED: the entire surplus is in the normal class; the three tumour classes match '
          'the paper exactly.')
else:
    print('DIFFERENT from the expected pattern - report what you actually see here, not the '
          'expectation above.')

## 3. Duplicate and leakage audit (LESSON 6)

MD5 over raw file bytes, so this is exact content identity - not a perceptual similarity measure.
A "leaked" file is one whose content appears in more than one split: the model can memorise it in
training and be rewarded for it at test time.

**Looks right**: ~153 files in duplicate groups, ~95% of them `normal`, and a nonzero train/test
leak count. This is the reason the `clean` split exists.

In [ ]:
from src.data_utils import add_hashes, find_duplicate_groups, audit_leakage

df = add_hashes(df)
print('unique content hashes:', df['hash'].nunique(), 'of', len(df), 'files')

In [ ]:
audit = audit_leakage(df)
for k, v in audit.items():
    if k.endswith('_rows'):
        continue
    print(f'{k}: {v}')

In [ ]:
dups = audit['duplicate_rows']
print('largest duplicate groups:')
top = dups['hash'].value_counts().head(5)
for h, n in top.items():
    rows = dups[dups['hash'] == h]
    print(f"  {h[:10]}... x{n}  class={rows['class'].iloc[0]}  "
          f"splits={sorted(set(rows['orig_split']))}")
    for f in rows['filename'].head(3):
        print(f'      {f}')

## 4. The two split variants

| Variant | Definition | Use |
|---|---|---|
| `faithful` | the dataset's own train/valid/test folders, duplicates included as-is | the paper-comparable split |
| `clean` | content-hash deduplicated, leakage-free, group-aware stratified 70/10/20 re-split | robustness / generalisation experiment |

**The `clean` variant is never a direct replication of the paper's number.** It is labelled as a
robustness experiment everywhere it appears.

`build_clean_split()` keeps exactly one representative per content hash, so no group can straddle
two splits - the leakage-free property holds by construction and is then asserted, not assumed.

In [ ]:
from src.data_utils import (build_faithful_split, build_clean_split, save_split,
                            split_counts, assert_no_leakage, imbalance_ratio)

faithful = build_faithful_split(df)
print('--- faithful ---')
print(split_counts(faithful).to_string())
print('train imbalance ratio (max/min):', round(imbalance_ratio(faithful, 'train'), 2))
print('saved to', save_split(faithful, 'faithful'))

In [ ]:
clean = build_clean_split(df)
print('--- clean (deduplicated) ---')
print(split_counts(clean).to_string())
print('train imbalance ratio (max/min):', round(imbalance_ratio(clean, 'train'), 2))
assert_no_leakage(clean)
print('no-leakage assertion: PASSED')
print('saved to', save_split(clean, 'clean'))

## 5. Class imbalance on `clean` (LESSON 7)

Deduplication removes far more `normal` images than tumour ones, so the clean split is materially
imbalanced. Inverse-frequency `class_weight` is therefore **on by default for `clean`** and off for
`faithful` (`USE_CLASS_WEIGHTS_BY_SPLIT` in `src/config.py`).

**Looks right**: a weight clearly above 1.0 for `normal` and below 1.0 for the larger tumour classes.

In [ ]:
from src.train_utils import class_weights_for

for name, sdf in (('faithful', faithful), ('clean', clean)):
    labels = sdf[sdf['split'] == 'train']['label'].values
    cw = class_weights_for(name, labels)
    print(f'{name}: class_weight =',
          {CLASS_NAMES[k]: round(v, 3) for k, v in cw.items()} if cw else 'None (by design)')

## 6. Results files (headers only) and the audit report

`init_results_files()` writes the three CSVs with **column headers and no rows**. Numbers only ever
arrive through `record_result()` from a notebook that actually trained something (LESSON 8).

In [ ]:
from src.evaluate_utils import init_results_files, load_results

created = init_results_files(overwrite=False)
print('created:', created if created else 'nothing new - files already exist')
for kind in ('canonical', 'experiment', 'ablation'):
    t = load_results(kind)
    print(f'  {kind:11s}: {len(t)} rows, {len(t.columns)} columns')

In [ ]:
lines = [
    '# Dataset audit',
    '',
    f'Data root: `{resolve_data_root()}`',
    '',
    '## Class counts (ours vs paper)',
    '',
    '```',
    cmp.to_string(),
    '```',
    '',
    f"Total ours={int(cmp['ours'].sum())}, paper={int(cmp['paper'].sum())}, "
    f"difference={int(cmp['difference'].sum())} "
    f"(all of it in `normal` if the CONFIRMED line printed above).",
    '',
    '## Duplication and leakage',
    '',
]
lines += [f'- {k}: {v}' for k, v in audit.items() if not k.endswith('_rows')]
lines += [
    '',
    '## Splits written',
    '',
    '### faithful (paper-comparable, duplicates kept)',
    '',
    '```', split_counts(faithful).to_string(), '```',
    '',
    '### clean (deduplicated, leakage-free - robustness experiment, NOT a replication)',
    '',
    '```', split_counts(clean).to_string(), '```',
    '',
]
path = REPORTS_DIR / 'dataset_audit.md'
path.write_text('\n'.join(lines))
print('wrote', path)
print('\nnext: 01_preprocessing_check.ipynb')